# رابط ناهمگام

0.6.0a2. داده‌های نمونه مصنوعی هستند؛ قیمت واقعی بازار نیستند.

## 1 - ورود کتابخانه‌ها

In [ ]:
import pydara
from pydara import tse
from pydara.tse.samples.extended import sample_extended_async_client
from tempfile import TemporaryDirectory
from pathlib import Path
print(pydara.__version__)

## 2 - تعریف توابع

In [ ]:
def show(frame):
    display(tse.select_columns(frame).head())
    return len(frame)

## 3 - تنظیمات

In [ ]:
SYMBOL = 'فولاد'
START = '1403-12-27'
END = '1403-12-28'

## 4 - اجرا

In [ ]:
with TemporaryDirectory(prefix='pydara-async-') as tmp:
    async with sample_extended_async_client(storage=True, data_dir=Path(tmp)/'db') as client:
        trades = await client.get_trades(SYMBOL, day='2025-03-18')
        job = await client.download([SYMBOL], start=START, end=END, update=False, checkpoint_dir=Path(tmp)/'jobs')
        again = await client.download([SYMBOL], start=START, end=END, update=False, checkpoint_dir=Path(tmp)/'jobs')
        await client.flush(timeout=10)
        print(await client.backup(Path(tmp)/'backup.sqlite'))
        assert again.requests.resumed.all()

## 5 - گزارش نتیجه

In [ ]:
show(trades)
display(job.requests)
print('Native async / shared download plan OK')